# Exploring factors associated with the popularity of the game

### A potential client is considering releasing the game within the scope of the following genres: *city builder, survival horror, rts, colony sim, tower defense*. We will be using the current data from Steam to analyse factors associated with the success of the games, in particular the time of the release and the pricing

> Data was sourced from Kaggle https://www.kaggle.com/datasets/fronkongames/steam-games-dataset on 12.07.2026, so will be reflective of all the releases and available data prior to this date

In [1]:
# Firstly, we will need to import all the required libraries to proceed with the data analysis
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

# Setting the option to see all the columns in the preview to ensure all information is captured
pd.set_option('display.max_columns', None)

In [2]:
# We will now read in the file
try:
    games_data = pd.read_csv("games.csv")
except FileNotFoundError:
    print("\n!ERROR!" \
    "\nThe file you're trying to load is not found. Make sure to check the spelling and " \
    "punctuation and try again")
    raise SystemExit("Stop right there! Please see the error message above!")

In [3]:
# Proceeding to previewing the data to estimate what can be deleted
games_data.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],['Korean'],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,yujingamesc@gmail.com,True,False,False,0,NaN,0,0,0,NaN,19,0,The game includes the following elements. 1. G...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],['English'],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [4]:
# Checking that the last column consists of only NaN
games_data["Movies"].isna().all()

np.True_

### From the preview we can clearly see the initial issue with the data, as the two columns merged into one (DiscountDCL count) causing a shift of the data. The dataframe also did not read the AppID correctly, leading to a skeweness of the dataframe by one column to the left causing the 'Movies' column to be completely NaN. We will need to address this before proceeding with the further analysis

In [5]:
# We will need to get the names of the columns to confirm this hypothesis and proceed with the 
# data transformation
cols = games_data.columns.tolist()
print(cols)

['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


In [6]:
# Proceeding with the fixing and separating the columns to reflect the data correctly
fixed = []
for col in cols:
    if col == 'DiscountDLC count':
        fixed += ['Discount', 'DLC count']
    else:
        fixed.append(col)

games_data = pd.read_csv('games.csv', header=0, names=fixed)
print(games_data.columns.to_list())

['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


In [7]:
# Checking that the fix worked and the data is now presented correctly
games_data.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],['Korean'],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,yujingamesc@gmail.com,True,False,False,0,NaN,0,0,0,NaN,19,0,The game includes the following elements. 1. G...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],['English'],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


Now that we've got all the columns aligned with the correct representative information we will proceed with further evaluating the data and ensuring the data is being shown correctly

In [8]:
games_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  str    
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  int64  
 6   Price                       125855 non-null  float64
 7   Discount                    125855 non-null  int64  
 8   DLC count                   125855 non-null  int64  
 9   About the game              117393 non-null  str    
 10  Supported languages         125855 non-null  str    
 11  Full audio languages        125855 non-null  str    
 12  Reviews                     12181 non-null   str    
 13  Header image             

We can see now the full information and the data types of the correct columns. We will now identify and proceed with dropping the columns that will not contribute to our data analysis. These columns are:
- AppID (no relevance)
- About the game (no relevance)
- Header image (no relevance)
- Website (no relevance)
- Support url (no relevance)
- Support email (no relevance)
- Metacritic url (no relevance)
- Score rank (not enough sufficient data to deal with missing values)
- Average playtime two weeks (want the all time data only)
- Median playtime two weeks (want the all time data only)
- Developers (no relevance)
- Publishers (no relevance)
- Tags (no relevance)
- Screenshots (no relevance)
- Movies (no relevance)
- Notes (no relevance)

In [9]:
# Dropping the columns
games_data_reduced = games_data.drop(columns=['AppID', 'About the game', 'Header image', 'Website', 
                                              'Support url', 'Support email', 'Metacritic url', 'Score rank',
                                              'Average playtime two weeks', 'Median playtime two weeks',
                                              'Developers', 'Publishers', 'Tags', 'Screenshots', 'Movies', 'Notes'])
games_data_reduced.head()

,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,Supported languages,Full audio languages,Reviews,Windows,Mac,Linux,Metacritic score,User score,Positive,Negative,Achievements,Recommendations,Average playtime forever,Median playtime forever,Categories,Genres
0,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,[],[],NaN,True,False,False,0,0,0,0,0,0,0,0,NaN,NaN
1,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,['English'],[],NaN,True,False,False,0,0,252,3,0,231,8,8,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure
2,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"['English', 'French', 'German', 'Russian']",[],NaN,True,True,False,0,0,21,3,0,0,0,0,"Single-player,Family Sharing",Casual
3,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,['Korean'],['Korean'],NaN,True,False,False,0,0,0,0,19,0,0,0,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation"
4,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,['English'],['English'],NaN,True,False,False,0,0,0,0,0,0,0,0,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access"


# Missing Values and Data quality inspection

Now that we have reduced the dataframe only to the relevant columns we can move onto dealing with the missing values in the columns

In [10]:
games_data_reduced.info()

<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Name                      125854 non-null  str    
 1   Release date              125855 non-null  str    
 2   Estimated owners          125855 non-null  str    
 3   Peak CCU                  125855 non-null  int64  
 4   Required age              125855 non-null  int64  
 5   Price                     125855 non-null  float64
 6   Discount                  125855 non-null  int64  
 7   DLC count                 125855 non-null  int64  
 8   Supported languages       125855 non-null  str    
 9   Full audio languages      125855 non-null  str    
 10  Reviews                   12181 non-null   str    
 11  Windows                   125855 non-null  bool   
 12  Mac                       125855 non-null  bool   
 13  Linux                     125855 non-null  bool   
 14 

From the cell above we can see that the columns 'Reviews', 'Categories' and 'Genres' are the ones with the majority of the missing values. There is one row with the missing 'Name'. We will proceed to exploring the quality of the existing data in these columns to come to a decision regarding how to handle the missing data.

In [12]:
print(f"The unique values for Reviews is: \n{games_data_reduced['Reviews'].unique()}")

The unique values for Reviews is: 
<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

It appears unclear on whether the reviews column is sourced as one review or combination of different reviews for the game, therefore due to the lack of this clarity it was decided to drop this column from the final analysis.

In [ ]:
games_data_reduced = games_data_reduced.drop(columns='Reviews')

In [19]:
games_data_reduced.head()

,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,Supported languages,Full audio languages,Windows,Mac,Linux,Metacritic score,User score,Positive,Negative,Achievements,Recommendations,Average playtime forever,Median playtime forever,Categories,Genres
0,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,[],[],True,False,False,0,0,0,0,0,0,0,0,NaN,NaN
1,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,['English'],[],True,False,False,0,0,252,3,0,231,8,8,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure
2,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"['English', 'French', 'German', 'Russian']",[],True,True,False,0,0,21,3,0,0,0,0,"Single-player,Family Sharing",Casual
3,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,['Korean'],['Korean'],True,False,False,0,0,0,0,19,0,0,0,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation"
4,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,['English'],['English'],True,False,False,0,0,0,0,0,0,0,0,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access"


> Plan to complete during the next session:
> 1. Finalise cleaning the data and reshape the dates into a correct format
> 2. Tokenise different genres and categories (needs further research on how to approach)
> 3. Proceed with multiple linear regression exploring the impact of different factors on the success of the game (measured by how many people own it & play it all time)

In [13]:
print(f"The unique values for Reviews is: \n{games_data_reduced['Categories'].unique()}")

The unique values for Reviews is: 
<ArrowStringArray>
[                                                                                                                                                                                                                         nan,
                                                                                                                                                               'Single-player,Steam Trading Cards,Steam Cloud,Family Sharing',
                                                                                                                                                                                               'Single-player,Family Sharing',
                                                                                                                                                                            'Single-player,Steam Achievements,Family Sharing',
                                                      

In [14]:
print(f"The unique values for Reviews is: \n{games_data_reduced['Genres'].unique()}")

The unique values for Reviews is: 
<ArrowStringArray>
[                                                                                             nan,
                                                                                      'Adventure',
                                                                                         'Casual',
                                                                        'Casual,Indie,Simulation',
                                                                            'Action,Early Access',
                                                                               'Action,Adventure',
                                                                            'Simulation,Strategy',
                                                                                   'Action,Indie',
                                                                                   'Casual,Indie',
                                                       

In [17]:
games_data_reduced['Genres'].notna().value_counts()

Genres
True     117432
False      8423
Name: count, dtype: int64

In [11]:
'''for column in games_data.columns:
    print(f"Unique values for {column} are: ")
    print(games_data[column].unique())
    print("----------------------------------------------")'''

'for column in games_data.columns:\n    print(f"Unique values for {column} are: ")\n    print(games_data[column].unique())\n    print("----------------------------------------------")'